In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import warnings
warnings.filterwarnings('ignore')

In [2]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split, GridSearchCV

In [3]:
# ============================================================
# STEP 1: LOAD DATA
# ============================================================
import pandas as pd

file_path = "dataset/NIFTY_50_COMPANIES.csv" 
df = pd.read_csv(file_path)

print("="*60)
print("BASIC INFORMATION")
print("="*60)

# Shape
print(f"Shape (Rows, Columns): {df.shape}")

# Column names
print("\nColumns:")
print(df.columns.tolist())

# Data types
print("\nData Types:")
print(df.dtypes)

# ============================================================
# STEP 2: DATE INFORMATION
# ============================================================
print("\n" + "="*60)
print("DATE INFORMATION")
print("="*60)

df['Date'] = pd.to_datetime(df['Date'], errors='coerce')

print(f"Start Date : {df['Date'].min()}")
print(f"End Date   : {df['Date'].max()}")
print(f"Total Unique Dates : {df['Date'].nunique()}")

# ============================================================
# STEP 3: TICKER INFORMATION
# ============================================================
print("\n" + "="*60)
print("TICKER INFORMATION")
print("="*60)

if 'Ticker' in df.columns:
    print(f"Total Unique Tickers : {df['Ticker'].nunique()}")
    print("Ticker List:")
    print(df['Ticker'].unique())

# ============================================================
# STEP 4: MISSING VALUES
# ============================================================
print("\n" + "="*60)
print("MISSING VALUES")
print("="*60)

missing = df.isnull().sum()
print(missing[missing > 0])

# ============================================================
# STEP 5: NUMERICAL SUMMARY
# ============================================================
print("\n" + "="*60)
print("NUMERICAL SUMMARY")
print("="*60)

print(df.describe())

# ============================================================
# STEP 6: SAMPLE DATA
# ============================================================
print("\n" + "="*60)
print("FIRST 5 ROWS")
print("="*60)
print(df.head())

print("\n" + "="*60)
print("LAST 5 ROWS")
print("="*60)
print(df.tail())

BASIC INFORMATION
Shape (Rows, Columns): (304543, 19)

Columns:
['Date', 'Adj Close', 'Close', 'High', 'Low', 'Open', 'Volume', 'Ticker', 'SMA_20', 'SMA_50', 'EMA_12', 'EMA_26', 'MACD', 'Signal_Line', 'RSI_14', 'BB_Mid', 'BB_Upper', 'BB_Lower', 'Daily_Return_%']

Data Types:
Date                  str
Adj Close         float64
Close             float64
High              float64
Low               float64
Open              float64
Volume              int64
Ticker                str
SMA_20            float64
SMA_50            float64
EMA_12            float64
EMA_26            float64
MACD              float64
Signal_Line       float64
RSI_14            float64
BB_Mid            float64
BB_Upper          float64
BB_Lower          float64
Daily_Return_%    float64
dtype: object

DATE INFORMATION
Start Date : 1997-07-01 00:00:00
End Date   : 2025-11-11 00:00:00
Total Unique Dates : 7109

TICKER INFORMATION
Total Unique Tickers : 51
Ticker List:
<StringArray>
[  'RELIANCE.NS',        'TCS.NS'

In [4]:
# ============================================================
# STEP 2: DATA CLEANING
# ============================================================
print("\n" + "=" * 60)
print("STEP 2: DATA CLEANING")
print("=" * 60)

before = len(df)
print(f"Shape before: {df.shape}")
null_counts = df.isnull().sum()
print(f"Null counts:\n{null_counts[null_counts > 0].to_string()}")
df = df.dropna()
print(f"Rows dropped (NaN from indicator lookback): {before - len(df)}")
print(f"Shape after : {df.shape}")


STEP 2: DATA CLEANING
Shape before: (304543, 19)
Null counts:
SMA_20             969
SMA_50            2499
RSI_14            2697
BB_Mid             969
BB_Upper           969
BB_Lower           969
Daily_Return_%      51
Rows dropped (NaN from indicator lookback): 4533
Shape after : (300010, 19)


In [5]:
# ============================================================
# STEP 3: FEATURE ENGINEERING
# ============================================================

print("\n" + "=" * 60)
print("STEP 3: FEATURE ENGINEERING")
print("=" * 60)

import numpy as np

# Ensure sorted properly
df = df.sort_values(['Ticker', 'Date']).reset_index(drop=True)

# ------------------------------------------------------------
# FEATURE ENGINEERING PER TICKER (PROFESSIONAL METHOD)
# ------------------------------------------------------------

# 1️⃣ Daily Return (if not already correct)
df['Return'] = df.groupby('Ticker')['Adj Close'].pct_change()

# 2️⃣ Lag Features
df['Return_Lag1'] = df.groupby('Ticker')['Return'].shift(1)
df['Return_Lag2'] = df.groupby('Ticker')['Return'].shift(2)
df['Return_Lag3'] = df.groupby('Ticker')['Return'].shift(3)

# 3️⃣ Rolling Volatility
df['Volatility_10'] = (
    df.groupby('Ticker')['Return']
      .rolling(10)
      .std()
      .reset_index(level=0, drop=True)
)

# 4️⃣ Trend Ratios
df['SMA_ratio'] = df['Close'] / df['SMA_20']
df['EMA_ratio'] = df['Close'] / df['EMA_12']

# 5️⃣ Momentum Features
df['MACD_diff'] = df['MACD'] - df['Signal_Line']
df['RSI_norm']  = df['RSI_14'] / 100

# 6️⃣ Bollinger Band Position
bb_width = (df['BB_Upper'] - df['BB_Lower']).replace(0, np.nan)
df['BB_position'] = (df['Close'] - df['BB_Lower']) / bb_width

# ------------------------------------------------------------
# Show Added Features
# ------------------------------------------------------------

eng_features = [
    'Return',
    'Return_Lag1',
    'Return_Lag2',
    'Return_Lag3',
    'Volatility_10',
    'SMA_ratio',
    'EMA_ratio',
    'MACD_diff',
    'RSI_norm',
    'BB_position'
]

print("Features added:")
print(eng_features)

print("\nFeature Statistics:")
print(df[eng_features].describe().round(4))


STEP 3: FEATURE ENGINEERING
Features added:
['Return', 'Return_Lag1', 'Return_Lag2', 'Return_Lag3', 'Volatility_10', 'SMA_ratio', 'EMA_ratio', 'MACD_diff', 'RSI_norm', 'BB_position']

Feature Statistics:
            Return  Return_Lag1  Return_Lag2  Return_Lag3  Volatility_10  \
count  299959.0000  299908.0000  299857.0000  299806.0000    299500.0000   
mean        0.0020       0.0020       0.0020       0.0020         0.0237   
std         0.2140       0.2141       0.2141       0.2141         0.2133   
min        -0.9909      -0.9909      -0.9909      -0.9909         0.0000   
25%        -0.0102      -0.0102      -0.0102      -0.0102         0.0125   
50%         0.0001       0.0001       0.0001       0.0001         0.0172   
75%         0.0114       0.0114       0.0114       0.0114         0.0242   
max       108.4930     108.4930     108.4930     108.4930        34.3460   

         SMA_ratio    EMA_ratio    MACD_diff     RSI_norm  BB_position  
count  300010.0000  300010.0000  3000

In [6]:
# ============================================================
# STEP 4: CREATE TARGET LABEL (PROFESSIONAL VERSION)
# ============================================================

print("\n" + "=" * 60)
print("STEP 4: CREATE TARGET LABEL")
print("=" * 60)

# Ensure sorted properly
df = df.sort_values(['Ticker', 'Date']).reset_index(drop=True)

# ------------------------------------------------------------
# 1️⃣ Create Future Return per ticker
# ------------------------------------------------------------

df['Future_Return'] = df.groupby('Ticker')['Return'].shift(-1)

# ------------------------------------------------------------
# 2️⃣ Define Signal using vectorized logic
# ------------------------------------------------------------

threshold = 0.005  # 0.5%

df['Signal'] = 0
df.loc[df['Future_Return'] >  threshold, 'Signal'] =  1
df.loc[df['Future_Return'] < -threshold, 'Signal'] = -1

# ------------------------------------------------------------
# 3️⃣ Drop rows where Future_Return is NaN (last row per stock)
# ------------------------------------------------------------

df = df.dropna(subset=['Future_Return'])

# ------------------------------------------------------------
# 4️⃣ Print Class Distribution
# ------------------------------------------------------------

print(f"Threshold: ±{threshold*100:.2f}%  |  BUY=1, HOLD=0, SELL=-1\n")

counts = df['Signal'].value_counts().sort_index()

label_map = {1:'BUY', 0:'HOLD', -1:'SELL'}

for k in counts.index:
    v = counts[k]
    print(f"{label_map[k]:5s} ({k:+d}): {v:7d}  ({v/len(df)*100:.2f}%)")


STEP 4: CREATE TARGET LABEL
Threshold: ±0.50%  |  BUY=1, HOLD=0, SELL=-1

SELL  (-1):  107601  (35.87%)
HOLD  (+0):   79875  (26.63%)
BUY   (+1):  112483  (37.50%)


In [7]:
# ============================================================
# STEP 5: SELECT FINAL FEATURES
# ============================================================

print("\n" + "=" * 60)
print("STEP 5: SELECT FINAL FEATURES")
print("=" * 60)

# Ensure properly sorted
df = df.sort_values(['Ticker', 'Date']).reset_index(drop=True)

# ------------------------------------------------------------
# Final Feature List
# ------------------------------------------------------------

features = [
    'Return_Lag1',
    'Return_Lag2',
    'Return_Lag3',
    'Volatility_10',
    'SMA_ratio',
    'EMA_ratio',
    'MACD_diff',
    'RSI_norm',
    'BB_position'
]

# ------------------------------------------------------------
# Remove rows with missing values in features or target
# ------------------------------------------------------------

df = df.dropna(subset=features + ['Signal'])

# ------------------------------------------------------------
# Define X and y
# ------------------------------------------------------------

X = df[features]
y = df['Signal']

print(f"Selected Features ({len(features)}):")
print(features)

print(f"\nFinal Dataset Shape:")
print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")

# Check for remaining NaNs
print("\nRemaining NaNs in X:")
print(X.isnull().sum().sum())


STEP 5: SELECT FINAL FEATURES
Selected Features (9):
['Return_Lag1', 'Return_Lag2', 'Return_Lag3', 'Volatility_10', 'SMA_ratio', 'EMA_ratio', 'MACD_diff', 'RSI_norm', 'BB_position']

Final Dataset Shape:
X shape: (299449, 9)
y shape: (299449,)

Remaining NaNs in X:
0


In [8]:
# ============================================================
# STEP 6: WALK-FORWARD VALIDATION (TimeSeriesSplit)
# ============================================================

print("\n" + "=" * 60)
print("STEP 6: WALK-FORWARD VALIDATION (TimeSeriesSplit)")
print("=" * 60)

from sklearn.model_selection import TimeSeriesSplit
import numpy as np

# Ensure sorted globally by Date
df = df.sort_values(['Date', 'Ticker']).reset_index(drop=True)

X = df[features]
y = df['Signal']

tscv = TimeSeriesSplit(n_splits=5)

fold = 1

for train_index, test_index in tscv.split(X):

    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    d_train = df['Date'].iloc[train_index]
    d_test  = df['Date'].iloc[test_index]

    print(f"\n📊 Fold {fold}")
    print(f"Train: {len(X_train):6,} rows  "
          f"{d_train.min().date()} → {d_train.max().date()}")

    print(f"Test : {len(X_test):6,} rows  "
          f"{d_test.min().date()} → {d_test.max().date()}")

    fold += 1


STEP 6: WALK-FORWARD VALIDATION (TimeSeriesSplit)

📊 Fold 1
Train: 49,909 rows  1997-09-22 → 2004-11-09
Test : 49,908 rows  2004-11-09 → 2009-06-10

📊 Fold 2
Train: 99,817 rows  1997-09-22 → 2009-06-10
Test : 49,908 rows  2009-06-10 → 2013-09-05

📊 Fold 3
Train: 149,725 rows  1997-09-22 → 2013-09-05
Test : 49,908 rows  2013-09-05 → 2017-11-29

📊 Fold 4
Train: 199,633 rows  1997-09-22 → 2017-11-29
Test : 49,908 rows  2017-11-29 → 2021-11-26

📊 Fold 5
Train: 249,541 rows  1997-09-22 → 2021-11-26
Test : 49,908 rows  2021-11-26 → 2025-11-10


In [9]:
# ============================================================
# STEP 7: FEATURE SCALING
# ============================================================

print("\n" + "=" * 60)
print("STEP 7: FEATURE SCALING (StandardScaler)")
print("=" * 60)

from sklearn.preprocessing import StandardScaler
import numpy as np

# Create scaler object
scaler = StandardScaler()

# ------------------------------------------------------------
# IMPORTANT:
# fit() ONLY on TRAIN data
# transform() on TRAIN and TEST
# ------------------------------------------------------------

X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print("✔ fit() on TRAIN only")
print("✔ transform() on TEST")
print("✔ No data leakage")

# ------------------------------------------------------------
# Verification
# ------------------------------------------------------------

print("\nPost-Scaling Check (Train Data):")
print(f"Mean ≈ {np.mean(X_train_sc):.6f}   (target ≈ 0)")
print(f"Std  ≈ {np.std(X_train_sc):.6f}    (target ≈ 1)")

print("\nShapes After Scaling:")
print(f"X_train_sc shape: {X_train_sc.shape}")
print(f"X_test_sc  shape: {X_test_sc.shape}")


STEP 7: FEATURE SCALING (StandardScaler)
✔ fit() on TRAIN only
✔ transform() on TEST
✔ No data leakage

Post-Scaling Check (Train Data):
Mean ≈ 0.000000   (target ≈ 0)
Std  ≈ 1.000000    (target ≈ 1)

Shapes After Scaling:
X_train_sc shape: (249541, 9)
X_test_sc  shape: (49908, 9)


In [10]:
# ============================================================
# STEP 8: TRAIN MULTIPLE SUPERVISED MODELS
# ============================================================

print("\n" + "=" * 60)
print("STEP 8: TRAIN MULTIPLE SUPERVISED MODELS")
print("=" * 60)

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier,
    GradientBoostingClassifier,
    AdaBoostClassifier,
    HistGradientBoostingClassifier
)
from sklearn.metrics import accuracy_score
import pandas as pd
import numpy as np

models = {

    # Linear
    "Logistic Regression":
        LogisticRegression(max_iter=1000, class_weight='balanced'),

    # Distance
    "KNN":
        KNeighborsClassifier(n_neighbors=5),

    # SVM
    "SVM (Linear)":
        SVC(class_weight='balanced', max_iter=5000),

    "SVM (RBF)":
        SVC(class_weight='balanced', max_iter=5000),

    # Tree
    "Decision Tree":
        DecisionTreeClassifier(max_depth=6, class_weight='balanced'),

    "Random Forest":
        RandomForestClassifier(
            n_estimators=200,
            max_depth=6,
            class_weight='balanced',
            random_state=42,
            n_jobs=-1
        ),

    # Boosting
    "Gradient Boosting":
        GradientBoostingClassifier(),

    "AdaBoost":
        AdaBoostClassifier(),

    "HistGradientBoosting":
        HistGradientBoostingClassifier()
}

results = {}

for name, model in models.items():

    print(f"\nTraining {name}...")

    if name in ["Decision Tree", "Random Forest",
                "Gradient Boosting", "AdaBoost",
                "HistGradientBoosting"]:
        
        model.fit(X_train, y_train)
        pred = model.predict(X_test)
    else:
        model.fit(X_train_sc, y_train)
        pred = model.predict(X_test_sc)

    acc = accuracy_score(y_test, pred)
    results[name] = acc

    print(f"{name} Accuracy: {acc:.4f}")

# ------------------------------------------------------------
# Final Comparison
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("MODEL COMPARISON (Accuracy)")
print("=" * 60)

for name, score in sorted(results.items(), key=lambda x: x[1], reverse=True):
    print(f"{name:25s}: {score:.4f}")


STEP 8: TRAIN MULTIPLE SUPERVISED MODELS

Training Logistic Regression...
Logistic Regression Accuracy: 0.3462

Training KNN...
KNN Accuracy: 0.3371

Training SVM (Linear)...
SVM (Linear) Accuracy: 0.3240

Training SVM (RBF)...
SVM (RBF) Accuracy: 0.3240

Training Decision Tree...
Decision Tree Accuracy: 0.3516

Training Random Forest...
Random Forest Accuracy: 0.3543

Training Gradient Boosting...
Gradient Boosting Accuracy: 0.3627

Training AdaBoost...
AdaBoost Accuracy: 0.3463

Training HistGradientBoosting...
HistGradientBoosting Accuracy: 0.3619

MODEL COMPARISON (Accuracy)
Gradient Boosting        : 0.3627
HistGradientBoosting     : 0.3619
Random Forest            : 0.3543
Decision Tree            : 0.3516
AdaBoost                 : 0.3463
Logistic Regression      : 0.3462
KNN                      : 0.3371
SVM (Linear)             : 0.3240
SVM (RBF)                : 0.3240


In [11]:
# ============================================================
# SELECT BEST MODEL + RETRAIN + EVALUATE
# ============================================================

from sklearn.metrics import classification_report, confusion_matrix

# ------------------------------------------------------------
# STEP 1: Find Best Model
# ------------------------------------------------------------

best_model_name = max(results, key=results.get)
print(f"\nBest Model: {best_model_name}")

best_model = models[best_model_name]

# ------------------------------------------------------------
# STEP 2: Retrain Best Model Properly
# ------------------------------------------------------------

tree_based_models = [
    "Decision Tree",
    "Random Forest",
    "Gradient Boosting",
    "AdaBoost",
    "HistGradientBoosting"
]

if best_model_name in tree_based_models:
    best_model.fit(X_train, y_train)
    pred = best_model.predict(X_test)
else:
    best_model.fit(X_train_sc, y_train)
    pred = best_model.predict(X_test_sc)

# ------------------------------------------------------------
# STEP 3: Evaluate
# ------------------------------------------------------------

print("\nClassification Report:\n")

print(classification_report(
    y_test,
    pred,
    target_names=['SELL(-1)', 'HOLD(0)', 'BUY(+1)'],
    labels=[-1, 0, 1]
))

cm = confusion_matrix(y_test, pred, labels=[-1, 0, 1])

print("Confusion Matrix (rows=Actual, cols=Predicted):")
print(f"{'':12s} SELL   HOLD   BUY")

for i, lab in enumerate(['SELL', 'HOLD', 'BUY']):
    print(f"  {lab:8s} {cm[i]}")


Best Model: Gradient Boosting

Classification Report:

              precision    recall  f1-score   support

    SELL(-1)       0.35      0.14      0.20     16271
     HOLD(0)       0.38      0.29      0.33     16080
     BUY(+1)       0.36      0.64      0.46     17557

    accuracy                           0.36     49908
   macro avg       0.36      0.36      0.33     49908
weighted avg       0.36      0.36      0.33     49908

Confusion Matrix (rows=Actual, cols=Predicted):
             SELL   HOLD   BUY
  SELL     [ 2241  3648 10382]
  HOLD     [1885 4627 9568]
  BUY      [ 2308  4016 11233]


In [21]:
# ============================================================
# STEP 9: MODEL COMPARISON + SMART SAVE (SWAP IF BETTER)
# ============================================================

import os, joblib
from sklearn.metrics import accuracy_score

print("\n" + "=" * 60)
print("STEP 9: MODEL COMPARISON + SMART PKL SAVE")
print("=" * 60)

for name, score in sorted(results.items(), key=lambda x: x[1], reverse=True):
    print(f"{name:25s}: {score:.4f}")

# ------------------------------------------------------------
# SELECT BEST MODEL
# ------------------------------------------------------------

best_model_name = max(results, key=results.get)
best_accuracy   = results[best_model_name]
print(f"\nBest Model This Run: {best_model_name}  (Acc = {best_accuracy:.4f})")

best_model = models[best_model_name]

tree_models = [
    "Decision Tree", "Random Forest",
    "Gradient Boosting", "AdaBoost", "HistGradientBoosting"
]

if best_model_name in tree_models:
    best_model.fit(X_train, y_train)
    best_pred = best_model.predict(X_test)
else:
    best_model.fit(X_train_sc, y_train)
    best_pred = best_model.predict(X_test_sc)

# ------------------------------------------------------------
# SMART PKL SWAP: only save if accuracy improved
# ------------------------------------------------------------

pkl_path = "best_model.pkl"
meta_path = "best_model_meta.pkl"

should_save = True

if os.path.exists(pkl_path) and os.path.exists(meta_path):
    old_meta = joblib.load(meta_path)
    old_acc  = old_meta.get("accuracy", 0)
    old_name = old_meta.get("model_name", "Unknown")
    print(f"\nExisting saved model : {old_name}  (Acc = {old_acc:.4f})")

    if best_accuracy > old_acc:
        print(f"✔ New model is BETTER → swapping pkl")
    else:
        print(f"✘ Existing model is same or better → keeping old pkl")
        should_save = False
else:
    print("\nNo existing model found → saving new model")

if should_save:
    joblib.dump(best_model, pkl_path)
    joblib.dump({
        "model_name": best_model_name,
        "accuracy": best_accuracy,
        "features": features
    }, meta_path)
    print(f"Saved: {pkl_path} + {meta_path}")

# ------------------------------------------------------------
# Save model comparison bar chart → output/PORTFOLIO/
# ------------------------------------------------------------

os.makedirs("output/PORTFOLIO", exist_ok=True)

sorted_results = dict(sorted(results.items(), key=lambda x: x[1], reverse=True))

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(list(sorted_results.keys()), list(sorted_results.values()), color='steelblue')
ax.set_xlabel('Accuracy')
ax.set_title('Model Comparison (Accuracy)')
ax.invert_yaxis()
for bar, val in zip(bars, sorted_results.values()):
    ax.text(bar.get_width() + 0.002, bar.get_y() + bar.get_height()/2,
            f'{val:.4f}', va='center', fontsize=9)
plt.tight_layout()
plt.savefig("output/PORTFOLIO/model_comparison.png", dpi=150)
plt.close()
print("\nSaved: output/PORTFOLIO/model_comparison.png")


STEP 9: MODEL COMPARISON + SMART PKL SAVE
Gradient Boosting        : 0.3627
HistGradientBoosting     : 0.3619
Random Forest            : 0.3543
Decision Tree            : 0.3516
AdaBoost                 : 0.3463
Logistic Regression      : 0.3462
KNN                      : 0.3371
SVM (Linear)             : 0.3240
SVM (RBF)                : 0.3240

Best Model This Run: Gradient Boosting  (Acc = 0.3627)

No existing model found → saving new model
Saved: best_model.pkl + best_model_meta.pkl

Saved: output/PORTFOLIO/model_comparison.png


In [22]:
# ============================================================
# CONFUSION MATRIX + CLASSIFICATION REPORT → output/PORTFOLIO/
# ============================================================

from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

print("\nClassification Report:\n")
print(classification_report(
    y_test, best_pred,
    target_names=['SELL(-1)', 'HOLD(0)', 'BUY(+1)'],
    labels=[-1, 0, 1]
))

cm = confusion_matrix(y_test, best_pred, labels=[-1, 0, 1])

fig, ax = plt.subplots(figsize=(6, 5))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['SELL', 'HOLD', 'BUY'])
disp.plot(ax=ax, cmap='Blues', values_format='d')
ax.set_title(f'Confusion Matrix — {best_model_name}')
plt.tight_layout()
plt.savefig("output/PORTFOLIO/confusion_matrix.png", dpi=150)
plt.close()
print("Saved: output/PORTFOLIO/confusion_matrix.png")


Classification Report:

              precision    recall  f1-score   support

    SELL(-1)       0.35      0.14      0.20     16271
     HOLD(0)       0.38      0.29      0.33     16080
     BUY(+1)       0.36      0.64      0.46     17557

    accuracy                           0.36     49908
   macro avg       0.36      0.36      0.33     49908
weighted avg       0.36      0.36      0.33     49908

Saved: output/PORTFOLIO/confusion_matrix.png


In [23]:
# ============================================================
# STEP 10: PORTFOLIO BACKTEST + GRAPHS → output/PORTFOLIO/
# ============================================================

print("\n" + "=" * 60)
print("STEP 10: PORTFOLIO BACKTEST + RISK METRICS")
print("=" * 60)

# Prepare test dataframe
df_test = df.iloc[test_index].copy().reset_index(drop=True)

best_pred_series = pd.Series(best_pred).reset_index(drop=True)
df_test = df_test.iloc[:len(best_pred_series)].copy()
df_test['Predicted_Signal'] = best_pred_series.values

df_test['Strategy_Return'] = df_test['Predicted_Signal'] * df_test['Future_Return']

tc = 0.001
df_test['Trade'] = (
    df_test.groupby('Ticker')['Predicted_Signal']
    .diff().abs().fillna(0)
)
df_test['Strategy_Return'] -= tc * df_test['Trade']

# Equal-weighted portfolio
daily_strategy = df_test.groupby('Date')['Strategy_Return'].mean()
daily_market   = df_test.groupby('Date')['Future_Return'].mean()

cum_strategy = (1 + daily_strategy).cumprod()
cum_market   = (1 + daily_market).cumprod()

sharpe = (daily_strategy.mean() / daily_strategy.std()) * np.sqrt(252) if daily_strategy.std() != 0 else 0
roll_max = cum_strategy.cummax()
drawdown = cum_strategy / roll_max - 1
max_dd   = drawdown.min()

final_s = cum_strategy.iloc[-1]
final_m = cum_market.iloc[-1]

print(f"  Strategy Total Return : {(final_s - 1) * 100:+.2f}%")
print(f"  Market  Total Return  : {(final_m - 1) * 100:+.2f}%")
print(f"  Sharpe Ratio          : {sharpe:.4f}")
print(f"  Max Drawdown          : {max_dd * 100:.2f}%")
print(f"  Test Period           : {df_test['Date'].min().date()} → {df_test['Date'].max().date()}")

# --- Cumulative Returns Plot ---
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(cum_strategy.index, cum_strategy.values, label='Strategy', linewidth=1.2)
ax.plot(cum_market.index, cum_market.values, label='Market (Buy & Hold)', linewidth=1.2, alpha=0.8)
ax.set_title('Portfolio Cumulative Returns — Strategy vs Market')
ax.set_xlabel('Date')
ax.set_ylabel('Growth of ₹1')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("output/PORTFOLIO/cumulative_returns.png", dpi=150)
plt.close()

# --- Drawdown Plot ---
fig, ax = plt.subplots(figsize=(12, 3))
ax.fill_between(drawdown.index, drawdown.values, 0, color='red', alpha=0.4)
ax.set_title('Portfolio Drawdown')
ax.set_ylabel('Drawdown')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("output/PORTFOLIO/drawdown.png", dpi=150)
plt.close()

print("\nSaved: output/PORTFOLIO/cumulative_returns.png")
print("Saved: output/PORTFOLIO/drawdown.png")


STEP 10: PORTFOLIO BACKTEST + RISK METRICS
  Strategy Total Return : -1.78%
  Market  Total Return  : +81.70%
  Sharpe Ratio          : -0.0095
  Max Drawdown          : -13.24%
  Test Period           : 2021-11-26 → 2025-11-10

Saved: output/PORTFOLIO/cumulative_returns.png
Saved: output/PORTFOLIO/drawdown.png


In [24]:
# ============================================================
# STEP 11: PER-COMPANY BACKTEST + GRAPHS → output/<company>/
# ============================================================

import os, numpy as np, pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier

print("\n" + "=" * 60)
print("STEP 11: PER-COMPANY BACKTEST + GRAPHS")
print("=" * 60)

all_tickers = sorted(df['Ticker'].unique())
print(f"Total tickers: {len(all_tickers)}\n")

company_summary = []

for ticker in all_tickers:
    # Clean company name for folder (strip .NS / .BO)
    company_name = ticker.replace('.NS', '').replace('.BO', '')
    out_dir = f"output/{company_name}"
    os.makedirs(out_dir, exist_ok=True)

    # --- Filter ---
    df_t = df[df['Ticker'] == ticker].copy().sort_values('Date').reset_index(drop=True)

    if len(df_t) < 100:
        print(f"  {ticker}: skipped (only {len(df_t)} rows)")
        continue

    X_t = df_t[features]
    y_t = df_t['Signal']

    split = int(len(df_t) * 0.8)

    X_tr, X_te = X_t.iloc[:split], X_t.iloc[split:]
    y_tr, y_te = y_t.iloc[:split], y_t.iloc[split:]

    # --- Train Random Forest ---
    model_t = RandomForestClassifier(
        n_estimators=200, max_depth=6,
        class_weight='balanced', random_state=42, n_jobs=-1
    )
    model_t.fit(X_tr, y_tr)
    pred_t = model_t.predict(X_te)

    acc_t = (pred_t == y_te.values).mean()

    # --- Backtest ---
    df_bt = df_t.iloc[split:].copy().reset_index(drop=True)
    df_bt['Predicted_Signal'] = pred_t
    df_bt['Strategy_Return']  = df_bt['Predicted_Signal'] * df_bt['Future_Return']

    tc = 0.001
    df_bt['Trade'] = df_bt['Predicted_Signal'].diff().abs().fillna(0)
    df_bt['Strategy_Return'] -= tc * df_bt['Trade']

    df_bt['Cum_Strategy'] = (1 + df_bt['Strategy_Return']).cumprod()
    df_bt['Cum_Market']   = (1 + df_bt['Future_Return']).cumprod()

    strat_ret = (df_bt['Cum_Strategy'].iloc[-1] - 1) * 100
    mkt_ret   = (df_bt['Cum_Market'].iloc[-1] - 1) * 100

    std_s = df_bt['Strategy_Return'].std()
    sharpe_t = (df_bt['Strategy_Return'].mean() / std_s) * np.sqrt(252) if std_s != 0 else 0

    rm = df_bt['Cum_Strategy'].cummax()
    dd = df_bt['Cum_Strategy'] / rm - 1
    max_dd_t = dd.min()

    company_summary.append({
        'Ticker': ticker,
        'Company': company_name,
        'Accuracy': acc_t,
        'Strategy_Return_%': strat_ret,
        'Market_Return_%': mkt_ret,
        'Sharpe': sharpe_t,
        'Max_Drawdown_%': max_dd_t * 100
    })

    # ============================================================
    # GRAPHS → output/<company>/
    # ============================================================

    # 1) Cumulative Returns
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(df_bt['Date'], df_bt['Cum_Strategy'], label='Strategy', linewidth=1.2)
    ax.plot(df_bt['Date'], df_bt['Cum_Market'], label='Market', linewidth=1.2, alpha=0.7)
    ax.set_title(f'{company_name} — Cumulative Returns')
    ax.set_xlabel('Date'); ax.set_ylabel('Growth of ₹1')
    ax.legend(); ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(f"{out_dir}/cumulative_returns.png", dpi=150)
    plt.close()

    # 2) Drawdown
    fig, ax = plt.subplots(figsize=(10, 3))
    ax.fill_between(df_bt['Date'], dd.values, 0, color='red', alpha=0.4)
    ax.set_title(f'{company_name} — Drawdown')
    ax.set_ylabel('Drawdown'); ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(f"{out_dir}/drawdown.png", dpi=150)
    plt.close()

    # 3) Signal Distribution
    fig, ax = plt.subplots(figsize=(6, 4))
    sig_counts = pd.Series(pred_t).map({-1:'SELL', 0:'HOLD', 1:'BUY'}).value_counts()
    sig_counts.reindex(['BUY','HOLD','SELL']).dropna().plot.bar(ax=ax, color=['green','gray','red'])
    ax.set_title(f'{company_name} — Predicted Signal Distribution')
    ax.set_ylabel('Count'); ax.set_xlabel('')
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.savefig(f"{out_dir}/signal_distribution.png", dpi=150)
    plt.close()

    # 4) Confusion Matrix
    from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
    cm_t = confusion_matrix(y_te, pred_t, labels=[-1, 0, 1])
    fig, ax = plt.subplots(figsize=(5, 4))
    disp = ConfusionMatrixDisplay(confusion_matrix=cm_t, display_labels=['SELL','HOLD','BUY'])
    disp.plot(ax=ax, cmap='Blues', values_format='d')
    ax.set_title(f'{company_name} — Confusion Matrix')
    plt.tight_layout()
    plt.savefig(f"{out_dir}/confusion_matrix.png", dpi=150)
    plt.close()

    print(f"  ✔ {company_name:15s}  Acc={acc_t:.4f}  Strat={strat_ret:+.1f}%  Mkt={mkt_ret:+.1f}%  Sharpe={sharpe_t:.3f}")

# ============================================================
# Summary Table
# ============================================================

df_summary = pd.DataFrame(company_summary)
df_summary = df_summary.sort_values('Strategy_Return_%', ascending=False).reset_index(drop=True)

print("\n" + "=" * 60)
print("COMPANY BACKTEST SUMMARY (sorted by Strategy Return)")
print("=" * 60)
print(df_summary.to_string(index=False))

# Save summary CSV
df_summary.to_csv("output/company_backtest_summary.csv", index=False)
print("\nSaved: output/company_backtest_summary.csv")


STEP 11: PER-COMPANY BACKTEST + GRAPHS
Total tickers: 51

  ✔ ADANIENT         Acc=0.3342  Strat=-74.4%  Mkt=+172.4%  Sharpe=-0.425
  ✔ ADANIGREEN       Acc=0.3051  Strat=-46.3%  Mkt=-44.5%  Sharpe=-0.804
  ✔ ADANIPORTS       Acc=0.3616  Strat=-18.6%  Mkt=+76.1%  Sharpe=-0.030
  ✔ APOLLOHOSP       Acc=0.3481  Strat=-65.0%  Mkt=+156.5%  Sharpe=-0.812
  ✔ ASIANPAINT       Acc=0.3760  Strat=-40.8%  Mkt=+15.3%  Sharpe=-0.631
  ✔ AXISBANK         Acc=0.3353  Strat=-56.1%  Mkt=+176.7%  Sharpe=-0.733
  ✔ BAJAJ-AUTO       Acc=0.3342  Strat=-63.0%  Mkt=+182.5%  Sharpe=-1.489
  ✔ BAJAJFINSV       Acc=0.3170  Strat=-59.3%  Mkt=+109.8%  Sharpe=-1.291
  ✔ BAJFINANCE       Acc=0.3325  Strat=-41.0%  Mkt=+91.0%  Sharpe=-0.584
  ✔ BHARTIARTL       Acc=0.3499  Strat=-25.9%  Mkt=+303.8%  Sharpe=-0.569
  ✔ BPCL             Acc=0.3343  Strat=-10.6%  Mkt=+167.0%  Sharpe=0.034
  ✔ BRITANNIA        Acc=0.3511  Strat=-64.7%  Mkt=+125.8%  Sharpe=-0.850
  ✔ CIPLA            Acc=0.3489  Strat=+31.8%  Mkt=+295.4%